# 02 · Visualization of benchmark runs

Reads `RUNS/all_results.csv` from `01_benchmark.ipynb` (and the saved structures in `RUNS/xyz/`). Figures:
per-case |ΔE| with the seed range, configuration-vs-configuration scatter, solve time, aqua, and 3-D overlays of the best conformer on the crystal. Colours follow the configuration in a fixed order (at most three per figure).

In [ ]:
import sys, os, json, subprocess, time
from pathlib import Path
import numpy as np, pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'jacobik').exists())
sys.path.insert(0, str(ROOT / 'src')); sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
from tools.jdev import bench, env
pd.set_option('display.width', 200); pd.set_option('display.max_rows', 200)
from IPython.display import HTML, display
import html as _html
import matplotlib.pyplot as plt
from tools.jdev import viz
RUNS = Path(os.environ.get('JDEV_RUNS') or sorted((ROOT / 'data' / 'benchmark_runs').glob('*/all_results.csv'), key=lambda p: p.stat().st_mtime)[-1].parent)   # latest; or set explicitly
ALL = pd.read_csv(RUNS / 'all_results.csv')
CONFIGS = list(dict.fromkeys(ALL.config))[:3]
SETS = [s for s in ('n48', 'ring10', 'heldout28', 'panel') if s in set(ALL.set)]
print(RUNS, CONFIGS, SETS, ALL.groupby(['set', 'config']).seed.nunique().to_dict())

## Per-case |ΔE| by case set (dot = seed mean, line = seed min–max; dashed = 100 / 300 kJ/mol)

In [ ]:
for s in SETS:
    viz.case_dots(ALL[ALL.set == s], CONFIGS, group='tmQM', title=f'{s}: per-case |ΔE| (tmQM)'); plt.show()

## Against the targets

In [ ]:
bench.targets(ALL[ALL.group == 'tmQM'])

## Configuration vs configuration (seed means; below the diagonal = the y configuration is better)

In [ ]:
if len(CONFIGS) >= 2:
    A, B = CONFIGS[0], CONFIGS[1]
    fig, axs = plt.subplots(1, len(SETS), figsize=(5.5 * len(SETS), 5.2), facecolor=viz.SURFACE, squeeze=False)
    for ax, s in zip(axs[0], SETS):
        viz.paired_scatter(ALL[ALL.set == s], A, B, ax=ax); ax.set_title(f'{s}: {B} vs {A}', loc='left', fontsize=11)
    plt.tight_layout(); plt.show()
else:
    print('one configuration in this run:', CONFIGS, '- nothing to pair')

In [ ]:
if len(CONFIGS) >= 3:
    A, B = CONFIGS[1], CONFIGS[2]
    fig, axs = plt.subplots(1, len(SETS), figsize=(5.5 * len(SETS), 5.2), facecolor=viz.SURFACE, squeeze=False)
    for ax, s in zip(axs[0], SETS):
        viz.paired_scatter(ALL[ALL.set == s], A, B, ax=ax); ax.set_title(f'{s}: {B} vs {A}', loc='left', fontsize=11)
    plt.tight_layout(); plt.show()

## Aqua (DOE, DFT references) and solve time

In [ ]:
if 'panel' in SETS:
    viz.case_dots(ALL[ALL.set == 'panel'], CONFIGS, group='aqua', title='DOE aqua: |ΔE|', refs=(20,)); plt.show()
for s in SETS:
    viz.time_dots(ALL[ALL.set == s], CONFIGS); plt.show()

## Summary table (seed-averaged per case, then over cases)

In [ ]:
# valid_only=False: the 'valid' flag compares coordination SITES with the crystal's donor ATOMS, so it wrongly
# drops every haptic complex (a Cp is 1 site, 5 atoms)
bench.summary(ALL, valid_only=False)

## 3-D overlays: best-scoring conformer (orange) on the crystal (blue), aligned on the mapped heavy atoms

In [ ]:
SET = SETS[0]
CASE, CONFIG, SEED = ALL[ALL.set == SET].case.iloc[0], CONFIGS[0], int(ALL.seed.min())   # set these to look at one structure
cands = sorted((RUNS / 'xyz').glob(f'{SET}*__{CONFIG}__s{SEED}'))   # '{SET}-sub' = a restricted run
assert cands, f'no saved structures for {SET}/{CONFIG}/seed {SEED} under {RUNS / "xyz"}'
xyz_dir = cands[0]; avail = sorted(m.name[:-9] for m in xyz_dir.glob('*_map.json'))
if CASE not in avail:
    print(f'{CASE} not in this run; available: {avail}'); CASE = avail[0]
out = RUNS / 'overlays'; out.mkdir(exist_ok=True)
info = viz.overlay_html(xyz_dir, CASE, out / f'{CASE}__{CONFIG}__s{SEED}.html'); print(info)
display(HTML(f'<iframe srcdoc="{_html.escape(Path(info["html"]).read_text())}" style="width:100%;height:620px;border:0"></iframe>'))

In [ ]:
# heavy-atom RMSD to the crystal for every saved structure (a geometry-only view next to |dE|).
# Caveat: not symmetry-corrected -- two equivalent groups swapped between solve and crystal (e.g. the two PPh3 of a
# trans complex) count as error, so a large RMSD with a small |dE| usually means such a swap, not a wrong structure.
rows = []
for d in sorted((RUNS / 'xyz').glob('*__*__s*')):
    tag, cfg, s = d.name.split('__')[:3]
    for m in d.glob('*_map.json'):
        case = m.name[:-9]
        r = viz.overlay_html(d, case, RUNS / 'overlays' / f'{case}__{cfg}__{s}.html')
        rows.append(dict(set=tag, config=cfg, seed=int(s[1:]), case=case, heavy_rmsd=r['heavy_rmsd']))
RMSD = pd.DataFrame(rows)
RMSD.pivot_table(index=['set', 'case'], columns='config', values='heavy_rmsd', aggfunc='mean').round(3)